# 05 - Multi-Qubit Gates

Explore two-qubit and three-qubit gates: CNOT, CZ, SWAP, Toffoli, Fredkin, and more.

**Concepts:** Controlled gates, SWAP family, Toffoli, Fredkin

In [ ]:
import quantsdk as qs
import math

## CNOT (CX) — The Entanglement Gate

In [ ]:
# CNOT flips target when control is |1>
for ctrl_state in ['0', '1']:
    circuit = qs.Circuit(2)
    if ctrl_state == '1':
        circuit.x(0)  # Set control to |1>
    circuit.cx(0, 1).measure_all()
    result = qs.run(circuit, shots=100, seed=42)
    print(f"CNOT with control={ctrl_state}: {result.counts}")

## Controlled Gates (CY, CZ, CH, CS, CSX)

In [ ]:
controlled_gates = [
    ("CY",  lambda c: c.cy(0, 1)),
    ("CZ",  lambda c: c.cz(0, 1)),
    ("CH",  lambda c: c.ch(0, 1)),
    ("CS",  lambda c: c.cs(0, 1)),
    ("CSX", lambda c: c.csx(0, 1)),
]

print("Controlled gates with control=|1>:")
for name, apply_gate in controlled_gates:
    circuit = qs.Circuit(2)
    circuit.x(0)  # Set control to |1>
    apply_gate(circuit)
    circuit.measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"  {name:4s}: {result.counts}")

## Controlled Rotations (CRX, CRY, CRZ, CP)

In [ ]:
# CRY with control=|1>
for theta in [0, math.pi/4, math.pi/2, math.pi]:
    circuit = qs.Circuit(2).x(0).cry(0, 1, theta).measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    p_target_1 = sum(c for bs, c in result.counts.items() if bs[1] == '1') / 1000
    print(f"CRY(theta={theta:.2f}): P(target=|1>) = {p_target_1:.3f}")

## SWAP Gate Family

In [ ]:
# SWAP: exchanges qubit states
circuit = qs.Circuit(2).x(0).swap(0, 1).measure_all()  # |10> -> |01>
result = qs.run(circuit, shots=100, seed=42)
print(f"SWAP |10> = {result.counts}")

# iSWAP: SWAP with a phase
circuit = qs.Circuit(2).x(0).iswap(0, 1).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"iSWAP |10> = {result.counts}")

# DCX (Double-CX)
circuit = qs.Circuit(2).x(0).dcx(0, 1).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"DCX |10> = {result.counts}")

## Three-Qubit Gates

In [ ]:
# Toffoli (CCX): AND gate — flips target only when both controls are |1>
for input_state in ['000', '010', '100', '110']:
    circuit = qs.Circuit(3)
    if input_state[0] == '1': circuit.x(0)
    if input_state[1] == '1': circuit.x(1)
    circuit.ccx(0, 1, 2).measure_all()
    result = qs.run(circuit, shots=100, seed=42)
    print(f"Toffoli |{input_state}> = {result.most_likely}")

print("\nToffoli acts as AND gate on the target qubit!")

In [ ]:
# Fredkin (CSWAP): Controlled-SWAP
# Swaps qubits 1,2 only when control (qubit 0) is |1>
circuit = qs.Circuit(3).x(0).x(1).cswap(0, 1, 2).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"Fredkin |110> = {result.most_likely} (swapped q1,q2)")

circuit = qs.Circuit(3).x(1).cswap(0, 1, 2).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"Fredkin |010> = {result.most_likely} (control=0, no swap)")

In [ ]:
# CCZ: Controlled-CZ
# Applies phase -1 only to |111>
circuit = qs.Circuit(3)
circuit.h(0).h(1).h(2)  # Superposition
circuit.ccz(0, 1, 2)    # Phase flip |111>
circuit.h(0).h(1).h(2)  # Interfere
circuit.measure_all()
result = qs.run(circuit, shots=2000, seed=42)
print(f"H-CCZ-H on |+++>: {result.counts}")